**Install required libraries for LangGraph and Google Cloud AI Platform.**


In [ ]:
#install the langgraph gcp components. This uses langchain-google-genai.
%pip install -U langchain_google_genai langgraph google-cloud-aiplatform

**Import necessary libraries for GCP authentication, BigQuery, LLM, LangGraph, and charting.**


In [ ]:
#basic libraries for gcp authentication
import sys
import os
from google.auth import default

#get libraries for bigquery
from google.cloud import bigquery
import json

#get the llm
from langchain_google_genai import ChatGoogleGenerativeAI

#langgraph related libraries
from typing import List, Dict, Any, TypedDict, Literal
from google.cloud.exceptions import GoogleCloudError

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

from langchain_core.runnables.graph import CurveStyle, MermaidDrawMethod, NodeStyles

from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from langgraph.prebuilt import ToolNode, tools_condition


#libraries needed for charting
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import base64
from io import BytesIO
import datetime
import re
from IPython.display import Image, display, Markdown

#other miscellaneous libraries
from google.cloud.exceptions import GoogleCloudError

**Configure Google Cloud project details and enable the AI Platform API.**


In [ ]:
#gcp project details. replace with your own google cloud project id and location
PROJECT_ID="adk-trials-493911"
LOCATION = "us-central1"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Google Gen AI configured for project '{PROJECT_ID}' in '{LOCATION}'")

**Authenticate with Google Cloud using `gcloud` commands.**


In [ ]:
#run these commands once either here or in the terminal to ensure that you have authorised a gcp account to be used
#replace with your own gcp login id and project id
!gcloud auth application-default set-quota-project your_project_id_on_gcp

!gcloud auth login 'your_email@xyz.com'
!gcloud config set account 'your_email@xyz.com'

**Create a new BigQuery dataset and load the `iris.csv` file into a table.**


In [ ]:
# Upload csv into a new dataset in bigquery under the given project
# This needs to be done once per dataset
# This is a simple scenario where only one csv file is being uploaded. This can easily be extended to handle multiple files or databases

bq_client = bigquery.Client(project=PROJECT_ID)
dataset_id = f"{PROJECT_ID}.iris" #create a new dataset
dataset = bigquery.Dataset(dataset_id)

# Specify the geographic location where the data should reside
dataset.location = "us-central1"

# Send the API request to create the dataset
dataset = bq_client.create_dataset(dataset, timeout=30)
print(f"Created dataset {bq_client.project}.{dataset.dataset_id}")

table_id = f"{dataset_id}.table_iris" #table name

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,  # Skip the header row
    autodetect=True,     # Automatically determine column names and types
)

file_path = "iris.csv"

with open(file_path, "rb") as source_file:
    job = bq_client.load_table_from_file(source_file, table_id, job_config=job_config)

job.result()  # Wait for the job to complete

table = bq_client.get_table(table_id)  # Make an API request to get table details
print(f"Loaded {table.num_rows} rows and {len(table.schema)} columns to {table_id}")

**Initialize the primary LLM (`gemini-2.5-flash`).**


In [ ]:
#Define the primary llm that will be used. You can use a llm from any provider.

model=ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=1)

**Test the LLM invocation and verify API keys in the environment variables.**


In [ ]:
test_message=[("user","how much is (4x10)-6?")]
print(model.invoke(test_message).text)

**Initialise the BigQuery client and fetch the database schema to seed the agent state.**


In [ ]:
# Initialise BigQuery Client
bq_client = bigquery.Client()
dataset_id = f"{PROJECT_ID}.iris"

# Gather schema details to provide context to the model
def get_database_schema(dataset_id: str) -> str:
    """Fetches schema information from BigQuery to seed the agent state."""
    schema_desc = []
    tables = bq_client.list_tables(dataset_id)
    for table in tables:
        table_ref = bq_client.get_table(table)
        schema_desc.append(f"Table: {table_ref.table_id}")
        for field in table_ref.schema:
            schema_desc.append(f"  - {field.name}: {field.field_type}")
    return "\n".join(schema_desc)


SCHEMA_CONTEXT = get_database_schema(dataset_id)

**Print the fetched database schema context.**


In [ ]:
print(SCHEMA_CONTEXT)

**Define the `AgentState` for the LangGraph workflow to hold conversational history, schema, queries, results and classifications.**


In [ ]:
# Define the Graph State where state info will be held
class AgentState(TypedDict):
    messages: List[BaseMessage]    # Conversational history
    schema: str                    # Database schema as context
    current_sql: str               # The SQL currently being evaluated
    iteration_count: int           # Loop counter to limit optimisation loops
    error_message: str | None      # Feedback from the evaluator (or None initially)
    query_results: List[Dict]      # Final data results from BigQuery
    final_answer: str              # User-facing summary analysis
    query_classification: Literal["SQL_QUERY", "GENERAL_QUERY"] | None # Classifies the user's query type
    has_chart_request: bool        # True if the user requested a chart

**Define the query classifier node to determine if a query is SQL-related or general, and detect chart requests.**


In [ ]:
# Query Classifier Node which determines if a query is sql related or a general query
def query_classifier_node(state: AgentState) -> Dict[str, Any]:
    """Classifies the user's query as either a SQL query or a general conversational question, and detects chart requests."""
    messages = state["messages"]

    classification_prompt = f"""You are a query classifier. Based on the user's latest message and the database schema, determine if it is a request for a SQL query / database-related question (SQL_QUERY) or a general conversational/off-topic question (GENERAL_QUERY). Also, determine if the user is asking for a chart or visualisation.
    Guidelines for Classification:
    1. SQL_QUERY:
       - Choose this for any request that asks for data, statistics, reports, or insights from the database.
       - IMPORTANT: Also choose this for any structural, metadata or schema-related questions about the dataset/database itself (e.g., "how many fields are there in the dataset?", "what are the columns in table_iris?", "show me the schema", "list all tables").
    2. GENERAL_QUERY:
       - Choose this ONLY for casual greetings, chit-chat or questions completely unrelated to the database, its schema or its data (e.g. "hello", "how are you?", "tell me a joke", "who created you?").
    Respond with a JSON object containing two keys:
    - 'query_type': 'SQL_QUERY' or 'GENERAL_QUERY'
    - 'chart_requested': true or false
    Example: {{ "query_type": "SQL_QUERY", "chart_requested": false }}
    Database Schema: {state['schema']}
    User's latest message: {messages[-1].content}
    """

    # Use LLM to classify the query
    response = model.invoke([HumanMessage(content=classification_prompt)])
    response_content = response.content.strip()

    
    json_string = response_content
    # Try to extract JSON from markdown block using regex
    json_string = re.sub(r'```json\s*|\s*```', '', response.content).strip()

    try:
        classification_output = json.loads(json_string)
        query_classification = classification_output.get("query_type", "GENERAL_QUERY").upper()
        has_chart_request = classification_output.get("chart_requested", False)
    except json.JSONDecodeError:
        print(f"Warning: Could not parse classifier response: {json_string}. Defaulting to GENERAL_QUERY.")
        query_classification = "GENERAL_QUERY"
        has_chart_request = False

    if query_classification not in ["SQL_QUERY", "GENERAL_QUERY"]:
        query_classification = "GENERAL_QUERY" # Fallback

    print(f"\n\n Query Classified as: {query_classification}, Chart Requested: {has_chart_request}") #Helps in debugging. Optional.
    return {"query_classification": query_classification, "has_chart_request": has_chart_request}

**Define the routing function to route the workflow based on the query classification.**


In [ ]:
# Routing function for the query classifier
def query_classifier_router(state: AgentState) -> Literal["SQL_QUERY", "GENERAL_QUERY"]:
    """Routes the graph based on the query classification."""
    #print(f"\n Routing based on classification: {state['query_classification']}") #Helps in debugging. Optional.
    return state["query_classification"]

**Define the general answer node to handle non-SQL conversational queries.**


In [ ]:
# General Answer Node
def general_answer_node(state: AgentState) -> Dict[str, Any]:
    """Generates a general, conversational answer for non-SQL queries."""
    messages = state["messages"]

    general_response_prompt = f"""You are a helpful assistant. The user's query was identified as a general conversational question, not requiring a database query.
    Respond to the user's latest message mentioning your inability to answer general queries. Mention that you specialise in queries about the `{dataset_id}` only.

    User's latest message: {messages[-1].content}
    """

    response = model.invoke([HumanMessage(content=general_response_prompt)])
    final_answer = response.content

    return {
        "final_answer": final_answer,
        "messages": messages + [AIMessage(content=final_answer)],
        "iteration_count": 0 # Reset counter
    }

**Define the generator/optimizer node to create or correct SQL queries based on user requests and evaluator feedback.**


In [ ]:
# The Generator / Optimiser Node. It generates the initial sql for the user query.
# The generated sql is sent to the evaluator. The Generator node also acts as the sql optimiser based on evaluator feedback

def generator_optimiser_node(state: AgentState) -> Dict[str, Any]:
    """Generates an initial SQL query or updates it based on evaluator error feedback."""
    messages = state["messages"]
    schema = state["schema"]
    current_sql = state["current_sql"]
    iteration = state.get("iteration_count", 0)
    error_message = state["error_message"]

    system_prompt = f"""You are an expert BigQuery SQL engineer.
Your task is to write valid BigQuery SQL queries based on user requests and the provided database schema.

CRITICAL RULES:
- Output ONLY valid, raw SQL inside markdown code blocks. Do not include introductory text.
- Do not use DML or DDL commands 
- Always use fully qualified table names: `{dataset_id}.table_name`
- Only use fields present in the schema.

### Database Schema:
{schema}
"""

    # Construct prompt based on whether it's an initial query or a correction loop
    llm_inputs = [SystemMessage(content=system_prompt)] + messages #user query + the above prompt , for the first time

    #if the previous sql was erroneous as deemed by the evaluator then add the error msg to the context
    if error_message:
        llm_inputs.append(
            HumanMessage(content=f"The previous query failed with this error:\n{error_message}\n\nPrevious SQL:\n```sql\n{current_sql}\n```\nCorrect the query syntax or logic and try again.")
        )

    response = model.invoke(llm_inputs)

    # Extract SQL text out of the markdown blocks cleanly
    raw_content = response.content
    sql_query = raw_content.split("```sql")[-1].split("```")[0].strip() if "```sql" in raw_content else raw_content.strip()

    return {
        "current_sql": sql_query,
        "iteration_count": iteration + 1
    }

**Define the evaluator node to validate the generated SQL query using BigQuery's dry run feature.**


In [ ]:
# The Evaluator Node. Validates, using bigquery's dry run feature, the generated sql query


def evaluator_node(state: AgentState) -> Dict[str, Any]:
    """Evaluates the generated SQL using BigQuery's Dry Run feature to check syntax validity."""
    sql = state["current_sql"]
    
    # Application-level check for DML/DDL commands
    # This rejects modifying commands before they hit the Cloud client.
    
    forbidden_pattern = r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|MERGE|TRUNCATE|GRANT|REVOKE)\b"
    if re.search(forbidden_pattern, sql, re.IGNORECASE):
        return {
            "error_message": "Security Violation: Only read-only SELECT queries are allowed. Modifying or structural commands are forbidden."
        }

    job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    try:
        # Dry run submits query to compiler but does not execute or incur cloud cost
        bq_client.query(sql, job_config=job_config)
        return {"error_message": ""} # Success, clear out previous errors
    except GoogleCloudError as e:
        # Failure detected, forward the exact engine logs to the optimiser
        return {"error_message": str(e.message)}


**Define the charting tool to generate visualizations from query results using Pandas and Seaborn.**


In [ ]:
# The charting tool. This is a simple pre-coded approach.
# A more complex and flexible approach would be to use a llm here to generate (and run) python code for charting in a sandboxed environment


@tool
def charting_tool(query_results: List[Dict], final_answer: str) -> str:
    """Generates a Seaborn or Matplotlib chart from BigQuery query results and embeds it in the final answer."""
    
    if not query_results:
        return {"final_answer": final_answer + "\n\n(No data available to plot.)"}

    # Prepare data for processing within this node
    processed_query_results = []
    for row in query_results:
        processed_row = {}
        for k, v in row.items():
            if isinstance(v, (pd.Timestamp, datetime.date, datetime.datetime)): # Ensure datetime objects are converted
                processed_row[k] = v.isoformat()
            else:
                processed_row[k] = v
        processed_query_results.append(processed_row)

    df = pd.DataFrame(processed_query_results)

    # Convert string columns that look like dates to datetime objects for plotting
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and df[col].str.contains(r'^\d{4}-\d{2}-\d{2}', na=False).any():
            try:
                df[col] = pd.to_datetime(df[col])
            except ValueError:
                pass # Not all strings might be dates, or mixed types. Handle gracefully.

    if len(df.columns) >= 2:
        x_col = df.columns[0]
        y_col = df.columns[1]

        plt.figure(figsize=(8, 4.5))

        # Dynamically determine plot type based on column data types
        if pd.api.types.is_datetime64_any_dtype(df[x_col]):
            sns.lineplot(x=x_col, y=y_col, data=df)
            plot_title = f'Time Series Chart for {x_col} vs {y_col}'
            plt.xlabel(x_col)
            plt.ylabel(y_col)
        elif (isinstance(df[x_col],pd.CategoricalDtype) or pd.api.types.is_object_dtype(df[x_col])) and pd.api.types.is_numeric_dtype(df[y_col]):
            sns.barplot(x=x_col, y=y_col, data=df)
            plot_title = f'Bar Chart for {x_col} vs {y_col}'
            plt.xlabel(x_col)
            plt.ylabel(y_col)
        elif (isinstance(df[y_col],pd.CategoricalDtype) or pd.api.types.is_object_dtype(df[y_col])) and pd.api.types.is_numeric_dtype(df[x_col]):
            sns.barplot(x=y_col, y=x_col, data=df) #swap the axes to get vertical bars
            plot_title = f'Bar Chart for {y_col} vs {x_col} - swap'
            plt.xlabel(y_col) #swap the labels accordingly
            plt.ylabel(x_col)
        elif pd.api.types.is_numeric_dtype(df[x_col]) and pd.api.types.is_numeric_dtype(df[y_col]):
            sns.scatterplot(x=x_col, y=y_col, data=df)
            plot_title = f'Scatter Plot for {x_col} vs {y_col}'
            plt.xlabel(x_col)
            plt.ylabel(y_col)
        else:
            # Fallback to a generic bar plot if other conditions don't match
            sns.barplot(x=x_col, y=y_col, data=df)
            plot_title = f'Chart for {x_col} vs {y_col} (Defaulting to Bar Plot)'
            plt.xlabel(x_col)
            plt.ylabel(y_col)

        plt.title(plot_title)
        plt.xticks(rotation=45)
        plt.tight_layout()

        # Save the plot to a BytesIO object
        buffer = BytesIO()
        plt.savefig(buffer, format='png')
        plt.close() # Close the plot to prevent it from displaying twice in some environments
        buffer.seek(0)
        chart_image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')

        # Embed the image in the final answer as a markdown image
        final_answer_with_chart = final_answer + f"\n\n![Chart](data:image/png;base64,{chart_image_base64})"
    else:
        final_answer_with_chart = final_answer + "\n\n(Not enough columns to generate a chart.)"

    return final_answer_with_chart

**Define the executor node to run the validated SQL query against BigQuery and format the results.**


In [ ]:
# Executor node. Executes final sql and formats answer


def executor_node(state: AgentState) -> Dict[str, Any]:
    """Executes validated SQL and formats data into a conversational response.
    May call a ToolNode to generate charts"""

   
    messages = state["messages"]
    # If the last message is a ToolMessage, 
    # the tool has already run and generated the chart! Return it directly.
    if messages and (isinstance(messages[-1], ToolMessage) or messages[-1].type == "tool"):
        return {
            "query_results": state.get("query_results", []),
            "final_answer": messages[-1].content,
            "messages": messages, # Do not append any new messages
            "error_message": "",
            "iteration_count": 0
        }

    has_chart_request = state["has_chart_request"]

    sql = state["current_sql"]

    # Check if we already have query results (avoids repeating query on 2nd turn)
    if not state.get("query_results"):
        query_job = bq_client.query(sql)
        results = [dict(row) for row in query_job.result()]
    else:
        results = state["query_results"]

    
    # Format the data into conversational text insights using the LLM
    answer_prompt = f"""You are a conversational data analyst. Analyse the following BigQuery result payload and answer the user's latest query clearly.

                        ### Executed SQL Query:
                        {sql}

                        ### Query Result Payload:
                        {results}

                        ### Chart Request Status:
                        Chart Requested: {has_chart_request}

                        INSTRUCTIONS FOR CHARTING:
                    - If has_chart_request is True, you MUST call the `charting_tool` to generate the requested chart.
                    - If has_chart_request is false, do not call the 'charting_tool'.
                    - Do NOT attempt to describe the chart visually yourself; the tool will handle rendering.

                    INSTRUCTIONS FOR DATA FORMATTING:
                    - **Tabular Format (When Appropriate):** If the query result payload contains structured rows, multiple records, or aggregates/summaries (e.g., a list of records, group-by groups, multiple statistics), you **MUST** present the data using a **Markdown Table** (e.g., `| Header 1 | Header 2 |` with clear alignments).
                    - **Descriptive Format:** If the result payload represents a single metric or scalar value (e.g., a single total count, average, or single row/value), present it in a clear, descriptive conversational sentence instead of a table.
                        """
    llm_inputs = [SystemMessage(content=answer_prompt)] + messages
    model_with_tools=model.bind_tools[(charting_tool)] # bind the charting tool here. The list format prevents unnecessary warning messages in the output
    response = model_with_tools.invoke(llm_inputs)

    # Extract clean text from LLM response.content blocks. Needed for llms bound with tools whose responses which are more complex than non tool llm responses
    raw_content = response.content
    if isinstance(raw_content, str):
        final_answer_content = raw_content
    elif isinstance(raw_content, list):
        # Extract and concatenate all text-type content blocks
        text_parts = []
        for block in raw_content:
            if isinstance(block, dict):
                if block.get("type") == "text":
                    text_parts.append(block.get("text", ""))
            elif isinstance(block, str):
                text_parts.append(block)
        final_answer_content = "".join(text_parts).strip()
    else:
        final_answer_content = str(raw_content)

    
    return {
        "query_results": results,
        "final_answer": final_answer_content,
        "messages": messages + [response],
        "error_message": "",
        "iteration_count": 0 # Reset counter for future multi-turn queries
    }

**Define routing logic to handle post-execution charting and conditional routing between the optimizer and executor nodes.**


In [ ]:
# Routing logic for execution or optimising

def routing_condition(state: AgentState) -> Literal["optimiser", "executor"]:
    """Determines whether to execute the query or route back to the optimiser."""
    if state["error_message"] and state.get("iteration_count", 0) < 4:
        print(f"🔄 Loop number {state['iteration_count']}: Error caught by Evaluator. Routing to Optimiser...")
        return "optimiser"
    return "executor"


**Build and compile the LangGraph workflow with nodes and conditional edges.**


In [ ]:
# Build and compile the Graph

workflow = StateGraph(AgentState)

# Add functional nodes
workflow.add_node("query_classifier", query_classifier_node)
workflow.add_node("general_answer", general_answer_node)
workflow.add_node("generator_optimiser", generator_optimiser_node)
workflow.add_node("evaluator", evaluator_node)
workflow.add_node("executor", executor_node)
workflow.add_node("charting_tool", ToolNode([charting_tool])) # Add the charting tool node

# Set the new entry point
workflow.set_entry_point("query_classifier")

# Add routing after query_classifier
workflow.add_conditional_edges(
    "query_classifier",
    query_classifier_router,
    {
        "SQL_QUERY": "generator_optimiser",
        "GENERAL_QUERY": "general_answer",
    }
)

# Add edge for general answer to end
workflow.add_edge("general_answer", END)

# Existing edges for SQL path
workflow.add_edge("generator_optimiser", "evaluator")

workflow.add_conditional_edges(
    "evaluator",
    routing_condition,
    {
        "optimiser": "generator_optimiser",
        "executor": "executor"
    }
)

# New routing after executor
workflow.add_conditional_edges(
    "executor",
    tools_condition,
    {
        "tools": "charting_tool",
        "__end__": END
    }
)

# Add edge from charting node to END
workflow.add_edge("charting_tool", "executor")

memory = MemorySaver()
app = workflow.compile(checkpointer=memory)


In [ ]:
#Visualise the graph

display(Image(app.get_graph().draw_mermaid_png()))

**Runs the graph in a basic UI setting**

In [ ]:
# A rudimentary UI which allows the user to specify queries and see the results as an ongoing conversation

# Maintain a persistent thread session ID for memory tracking
THREAD_CONFIG = {"configurable": {"thread_id": "analytics_session_01"}}

current_session_state = {
    "messages": [],
    "schema": SCHEMA_CONTEXT,
    "current_sql": "",
    "iteration_count": 0,
    "error_message": None,
    "query_results": [],
    "final_answer": "",
    "query_classification": None,
    "has_chart_request": False # Initialize the new flag
}

while True:
    user_query = input("\n==> Ask your question here (q/Q to quit): ") #A simple input box where the user specifies the query

    if user_query in {"q", "Q"}:
        break

    # Append the user's message for the current turn
    # Reset transient fields for a new query, but keep messages for context if THREAD_CONFIG works
    current_session_state["messages"].append(HumanMessage(content=user_query))
    current_session_state["current_sql"] = ""
    current_session_state["iteration_count"] = 0
    current_session_state["error_message"] = None
    current_session_state["query_results"] = []
    current_session_state["final_answer"] = "" # Reset final answer for a new query
    current_session_state["query_classification"] = None # Reset new field
    current_session_state["has_chart_request"] = False # Reset chart request for a new query

    # Invoke the graph directly to get the final state
    final_state = app.invoke(current_session_state, config=THREAD_CONFIG)

    # Update current_session_state with the final state
    current_session_state = final_state.copy()

    display(Markdown(f" #### Query: {user_query}"))
    # Display the final answer or chart once
    if current_session_state["final_answer"]:
        display(Markdown(f" #### Final Answer: {current_session_state['final_answer']}"))

**Optional debug code**

In [ ]:
# If detailed node info is needed, for example to debug the graph, create much more detailed output messages per node
# A much, much more sophisticated option is Langsmith, if you have access to it.

# Maintain a persistent thread session ID for memory tracking
THREAD_CONFIG = {"configurable": {"thread_id": "analytics_session_01"}}

current_session_state = {
    "messages": [],
    "schema": SCHEMA_CONTEXT,
    "current_sql": "",
    "iteration_count": 0,
    "error_message": None,
    "query_results": [],
    "final_answer": "",
    "query_classification": None,
    "has_chart_request": False # Initialize the new flag
}

from IPython.display import display, Markdown

while True:
    user_query = input("\n==> Ask your question here (q/Q to quit): ") #A simple input box where the user specifies the query

    if user_query in {"q", "Q"}:
        break

    # Append the user's message for the current turn
    # Reset transient fields for a new query, but keep messages for context if THREAD_CONFIG works
    current_session_state["messages"].append(HumanMessage(content=user_query))
    current_session_state["current_sql"] = ""
    current_session_state["iteration_count"] = 0
    current_session_state["error_message"] = None
    current_session_state["query_results"] = []
    current_session_state["final_answer"] = "" # Reset final answer for a new query
    current_session_state["query_classification"] = None # Reset new field
    current_session_state["has_chart_request"] = False # Reset chart request for a new query

    print(f"\n--- Starting new turn for query: '{user_query}' ---")

    # Store the state *before* the stream begins, for comparison for new AI messages
    prev_state_for_comparison = current_session_state.copy()

    for step_num, state in enumerate(app.stream(current_session_state, config=THREAD_CONFIG, stream_mode="values")):
        print(f"\n------ State after Node Execution (Step {step_num + 1}) ------")
        snapshot=app.get_state(THREAD_CONFIG)
        #node_executed = snapshot.metadata.get("source") # this doesnt seem to work properly

        # Fallback to heuristics to figure out the node
        node_inference = "Unknown Node"
        if state.get("query_classification") != prev_state_for_comparison.get("query_classification"):
            node_inference = "Query Classifier Node"
        elif state["current_sql"] != prev_state_for_comparison["current_sql"]:
            node_inference = "Generator/Optimiser Node"
        elif state["error_message"] != prev_state_for_comparison["error_message"]:
            node_inference = "Evaluator Node"
        elif state["final_answer"] != prev_state_for_comparison["final_answer"]:
            if state.get("query_classification") == "GENERAL_QUERY":
                node_inference = "General Answer Node"
            elif state.get("has_chart_request") == True and "![Chart]" in state["final_answer"]:
                node_inference = "Charting Node"
            else:
                node_inference = "Executor Node"
        print(f"****Node Inference:**** {node_inference}")

        print(f" ## Query Classification: {state.get('query_classification')}")
        print(f" ## Has Chart Request: {state.get('has_chart_request')}")

        print(f" ## Current SQL :")
        if state['current_sql']:
            print(f"```sql\n{state['current_sql']}\n```")
        else:
            print("  (No SQL generated yet)")

        print(f" ## Evaluation Feedback: {state['error_message']}")
        print(f" ## Iteration Count: {state['iteration_count']}")

        # Check for new AIMessage by comparing message list lengths
        new_ai_message = None
        if len(state["messages"]) > len(prev_state_for_comparison["messages"]):
            if isinstance(state["messages"][-1], AIMessage):
                new_ai_message = state["messages"][-1].content

        if new_ai_message:
            print(f" ## AI Message: {new_ai_message}")
        else:
            print(" ## AI Message: (None in this step)")

        if state["final_answer"]:
            # Use IPython.display.Markdown to render the final answer, especially if it contains images
            display(Markdown(f" ## Final Answer: {state['final_answer']}"))
        else:
            print(" ## Final Answer: (Not yet available)")

        # Update prev_state_for_comparison for the next iteration
        prev_state_for_comparison = state.copy()

    # After the loop, update current_session_state with the final state seen
    if 'state' in locals(): # Ensure 'state' was assigned at least once
        current_session_state = state.copy()
    else:
        # If no steps were executed (e.g., empty graph), use the initial state as final.
        current_session_state = prev_state_for_comparison.copy()